# 01 - Google Earth Engine Data Export

This notebook exports Sentinel-1 SAR and Sentinel-2 optical data from Google Earth Engine directly to local files.

**Outputs:**
- Sentinel-1 GRD: Monthly median VV backscatter (dB), 10m resolution
- Sentinel-2 SR: Monthly median (B4, B8 for NDVI), cloud-masked, 10m resolution
- ESA WorldCover wetland mask (10m, 2021)

**Study Area:**
- Bounding box: (99.98Â°E, 20.18Â°N) to (100.05Â°E, 20.25Â°N)
- Wiang Nong Lom wetland, Chiang Rai Province, Thailand

**Time Period:** 2019-01-01 to 2025-12-31

**Download method:** Direct to local via `geemap.ee_export_image()`

## Setup and Authentication

In [ ]:
import ee
import geemap
import os
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from pathlib import Path
import time

In [ ]:
# Authenticate and initialize Earth Engine
# Run this once if not already authenticated
# ee.Authenticate()

ee.Initialize(project='wiang-nom-lom')

## Define Study Area and Parameters

In [ ]:
# Study area bounding box
# Wiang Nong Lom wetland, Chiang Rai Province, Thailand
BBOX = [99.98, 20.18, 100.05, 20.25]  # [min_lon, min_lat, max_lon, max_lat]

# Create geometry
geometry = ee.Geometry.Rectangle(BBOX)

# Time range
START_DATE = '2019-01-01'
END_DATE = '2026-09-01'

# Export settings
EXPORT_SCALE = 10  # meters
EXPORT_CRS = 'EPSG:32647'  # UTM Zone 47N (covers this area of Thailand)

# Local output directories
S1_OUTPUT_DIR = Path('data/raw/sentinel1')
S2_OUTPUT_DIR = Path('data/raw/sentinel2')
S1_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
S2_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Generate list of months to process
def generate_month_list(start_date, end_date):
    """Generate list of (year, month) tuples between start and end dates."""
    start = datetime.strptime(start_date, '%Y-%m-%d')
    end = datetime.strptime(end_date, '%Y-%m-%d')
    
    months = []
    current = start
    while current <= end:
        months.append((current.year, current.month))
        current += relativedelta(months=1)
    
    return months

month_list = generate_month_list(START_DATE, END_DATE)
print(f"Total months to process: {len(month_list)}")
print(f"From {month_list[0]} to {month_list[-1]}")

## Visualize Study Area

In [ ]:
# Create interactive map
Map = geemap.Map(center=[20.215, 100.015], zoom=13)
Map.addLayer(geometry, {'color': 'red'}, 'Study Area')
Map

---
## Sentinel-1 SAR Export

Export monthly median VV backscatter (in dB) from Sentinel-1 GRD.

In [ ]:
def get_s1_monthly_composite(year, month, geometry):
    """
    Get monthly median Sentinel-1 VV backscatter composite.
    
    Parameters:
    -----------
    year : int
    month : int
    geometry : ee.Geometry
    
    Returns:
    --------
    ee.Image : Monthly median VV backscatter in dB
    """
    # Define date range for the month
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, 'month')
    
    # Filter Sentinel-1 GRD collection
    s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
          .filterBounds(geometry)
          .filterDate(start, end)
          .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
          .filter(ee.Filter.eq('instrumentMode', 'IW'))
          .select('VV'))
    
    # Compute monthly median
    composite = s1.median().clip(geometry)
    
    # Add metadata
    composite = composite.set({
        'year': year,
        'month': month,
        'system:time_start': start.millis()
    })
    
    return composite

In [ ]:
# Test: visualize a single month
test_composite = get_s1_monthly_composite(2020, 6, geometry)

Map2 = geemap.Map(center=[20.215, 100.015], zoom=13)
Map2.addLayer(test_composite, {'min': -25, 'max': 0, 'palette': ['blue', 'white', 'green']}, 'S1 VV June 2020')
Map2.addLayer(geometry, {'color': 'red'}, 'Study Area', opacity=0.5)
Map2

In [ ]:
def download_s1_monthly(year, month, geometry, output_dir, scale, crs):
    """
    Download Sentinel-1 monthly composite directly to local file.
    """
    composite = get_s1_monthly_composite(year, month, geometry)
    
    filename = f'S1_VV_{year}{month:02d}.tif'
    filepath = output_dir / filename
    
    # Skip if already downloaded
    if filepath.exists():
        print(f"  Skipping {filename} (already exists)")
        return filepath
    
    geemap.ee_export_image(
        composite,
        filename=str(filepath),
        scale=scale,
        region=geometry,
        crs=crs,
        file_per_band=False
    )
    
    return filepath

In [ ]:
# Download all Sentinel-1 monthly composites directly to local
# Skips files that already exist

s1_files = []
skipped = 0
downloaded = 0
failed = []

for i, (year, month) in enumerate(month_list):
    filepath = S1_OUTPUT_DIR / f'S1_VV_{year}{month:02d}.tif'
    
    if filepath.exists():
        s1_files.append(filepath)
        skipped += 1
        continue
    
    print(f"[{i+1}/{len(month_list)}] Downloading S1_VV_{year}{month:02d}...", end=" ")
    try:
        filepath = download_s1_monthly(year, month, geometry, S1_OUTPUT_DIR, EXPORT_SCALE, EXPORT_CRS)
        s1_files.append(filepath)
        downloaded += 1
        print("Done")
    except Exception as e:
        failed.append(f"{year}{month:02d}")
        print(f"FAILED: {e}")

print(f"\nS1 summary: {skipped} skipped, {downloaded} downloaded, {len(failed)} failed")
if failed:
    print(f"Failed months: {failed}")

---
## Sentinel-2 Optical Export

Export monthly median Sentinel-2 surface reflectance (B4-Red, B8-NIR) with cloud masking for NDVI calculation.

In [ ]:
def mask_s2_clouds(image):
    """
    Mask clouds in Sentinel-2 SR image using the QA60 band.
    """
    qa = image.select('QA60')
    
    # Bits 10 and 11 are clouds and cirrus
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    
    # Both flags should be zero for clear conditions
    mask = (qa.bitwiseAnd(cloud_bit_mask).eq(0)
            .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0)))
    
    return image.updateMask(mask)

In [ ]:
def get_s2_monthly_composite(year, month, geometry):
    """
    Get monthly median Sentinel-2 composite (B4, B8 for NDVI).
    
    Parameters:
    -----------
    year : int
    month : int
    geometry : ee.Geometry
    
    Returns:
    --------
    ee.Image : Monthly median B4 and B8 bands (scaled to 0-1 reflectance)
    """
    # Define date range for the month
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, 'month')
    
    # Filter Sentinel-2 SR collection
    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(geometry)
          .filterDate(start, end)
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 50))  # Pre-filter cloudy scenes
          .map(mask_s2_clouds)
          .select(['B4', 'B8']))  # Red and NIR bands for NDVI
    
    # Compute monthly median and scale to 0-1 reflectance
    composite = s2.median().divide(10000).clip(geometry)
    
    # Add metadata
    composite = composite.set({
        'year': year,
        'month': month,
        'system:time_start': start.millis()
    })
    
    return composite

In [ ]:
# Test: visualize a single month (compute NDVI for visualization)
test_s2 = get_s2_monthly_composite(2020, 6, geometry)
test_ndvi = test_s2.normalizedDifference(['B8', 'B4']).rename('NDVI')

Map3 = geemap.Map(center=[20.215, 100.015], zoom=13)
Map3.addLayer(test_ndvi, {'min': -0.2, 'max': 0.8, 'palette': ['brown', 'yellow', 'green', 'darkgreen']}, 'NDVI June 2020')
Map3.addLayer(geometry, {'color': 'red'}, 'Study Area', opacity=0.5)
Map3

In [ ]:
def download_s2_monthly(year, month, geometry, output_dir, scale, crs):
    """
    Download Sentinel-2 monthly composite directly to local file.
    """
    composite = get_s2_monthly_composite(year, month, geometry)
    
    filename = f'S2_{year}{month:02d}.tif'
    filepath = output_dir / filename
    
    # Skip if already downloaded
    if filepath.exists():
        print(f"  Skipping {filename} (already exists)")
        return filepath
    
    geemap.ee_export_image(
        composite,
        filename=str(filepath),
        scale=scale,
        region=geometry,
        crs=crs,
        file_per_band=False
    )
    
    return filepath

In [ ]:
# Download all Sentinel-2 monthly composites directly to local
# Skips files that already exist

s2_files = []
skipped = 0
downloaded = 0
failed = []

for i, (year, month) in enumerate(month_list):
    filepath = S2_OUTPUT_DIR / f'S2_{year}{month:02d}.tif'
    
    if filepath.exists():
        s2_files.append(filepath)
        skipped += 1
        continue
    
    print(f"[{i+1}/{len(month_list)}] Downloading S2_{year}{month:02d}...", end=" ")
    try:
        filepath = download_s2_monthly(year, month, geometry, S2_OUTPUT_DIR, EXPORT_SCALE, EXPORT_CRS)
        s2_files.append(filepath)
        downloaded += 1
        print("Done")
    except Exception as e:
        failed.append(f"{year}{month:02d}")
        print(f"FAILED: {e}")

print(f"\nS2 summary: {skipped} skipped, {downloaded} downloaded, {len(failed)} failed")
if failed:
    print(f"Failed months: {failed}")

---
## Verify Downloads

In [ ]:
# Verify downloaded files
s1_downloaded = sorted(S1_OUTPUT_DIR.glob('S1_VV_*.tif'))
s2_downloaded = sorted(S2_OUTPUT_DIR.glob('S2_*.tif'))

print(f"Sentinel-1 files: {len(s1_downloaded)}")
print(f"Sentinel-2 files: {len(s2_downloaded)}")

# Check for missing months
expected_months = set(f"{y}{m:02d}" for y, m in month_list)
s1_months = set(f.stem.split('_')[-1] for f in s1_downloaded)
s2_months = set(f.stem.split('_')[-1] for f in s2_downloaded)

s1_missing = expected_months - s1_months
s2_missing = expected_months - s2_months

if s1_missing:
    print(f"\nMissing S1 months: {sorted(s1_missing)}")
if s2_missing:
    print(f"\nMissing S2 months: {sorted(s2_missing)}")
if not s1_missing and not s2_missing:
    print("\nAll months downloaded successfully.")

## OpenStreetMap Wetland Mask

Download wetland/water body boundaries from OpenStreetMap via the Overpass API.
OSM has hand-traced boundaries for specific water bodies, which can be more accurate than satellite-derived classifications like ESA WorldCover.

The polygons are rasterized to match the Sentinel raster grid (10m, EPSG:32647).

In [ ]:
import requests
import rasterio
from rasterio.features import rasterize as rio_rasterize
from shapely.geometry import LineString, Polygon, MultiPolygon, mapping
from shapely.ops import linemerge, unary_union, polygonize
from pyproj import Transformer

# Query Overpass API for water/wetland features in study area
overpass_url = "https://overpass-api.de/api/interpreter"

# Use traditional out body + recurse down (lighter than out geom)
overpass_query = """
[out:json][timeout:120];
(
  way["natural"="water"](20.18,99.98,20.25,100.05);
  relation["natural"="water"](20.18,99.98,20.25,100.05);
  way["natural"="wetland"](20.18,99.98,20.25,100.05);
  relation["natural"="wetland"](20.18,99.98,20.25,100.05);
);
out body;
>;
out skel qt;
"""

print("Querying Overpass API for water/wetland features...")
response = requests.post(overpass_url, data={'data': overpass_query})
print(f"Response status: {response.status_code}, length: {len(response.text)} chars")

if response.status_code != 200:
    print(f"ERROR: {response.text[:500]}")
    raise RuntimeError("Overpass API request failed")

osm_data = response.json()

# Build node lookup: id -> (lon, lat)
nodes = {}
ways = {}
relations = []

for el in osm_data['elements']:
    if el['type'] == 'node':
        nodes[el['id']] = (el['lon'], el['lat'])
    elif el['type'] == 'way':
        ways[el['id']] = el
    elif el['type'] == 'relation':
        relations.append(el)

print(f"Parsed: {len(nodes)} nodes, {len(ways)} ways, {len(relations)} relations")

# Build polygons from ways
polygons = []

for wid, way in ways.items():
    tags = way.get('tags', {})
    # Only process ways that have water/wetland tags (not member-only ways)
    if 'natural' not in tags:
        continue
    
    nds = way.get('nodes', [])
    coords = [nodes[nid] for nid in nds if nid in nodes]
    
    if len(coords) >= 4 and coords[0] == coords[-1]:
        try:
            p = Polygon(coords)
            if p.is_valid and p.area > 0:
                polygons.append(p)
                name = tags.get('name', tags.get('name:en', 'unnamed'))
                print(f"  way/{wid}: {name} - {tags.get('natural', '')} ({p.area:.6f} sq deg)")
        except:
            pass

# Build polygons from relations (multipolygons)
for rel in relations:
    tags = rel.get('tags', {})
    name = tags.get('name', tags.get('name:en', 'unnamed'))
    
    outer_lines = []
    for member in rel.get('members', []):
        if member.get('role') in ('outer', '') and member['type'] == 'way':
            ref_way = ways.get(member['ref'])
            if ref_way:
                nds = ref_way.get('nodes', [])
                coords = [nodes[nid] for nid in nds if nid in nodes]
                if len(coords) >= 2:
                    outer_lines.append(LineString(coords))
    
    if outer_lines:
        merged = linemerge(outer_lines)
        for p in polygonize(merged):
            if p.is_valid and p.area > 0:
                polygons.append(p)
                print(f"  relation/{rel['id']}: {name} - {tags.get('natural', '')} ({p.area:.6f} sq deg)")

print(f"\nTotal: {len(polygons)} valid polygons")

if len(polygons) == 0:
    raise RuntimeError("No polygons found! Check the bbox or OSM tags.")

# Merge all polygons into single geometry
wetland_geom = unary_union(polygons)
print(f"Merged geometry: {wetland_geom.geom_type}")

# Visualize on map
geojson_data = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "geometry": mapping(wetland_geom),
        "properties": {"name": "OSM Wetland Mask"}
    }]
}

MapOSM = geemap.Map(center=[20.215, 100.015], zoom=13)
MapOSM.add_geojson(geojson_data, style={'color': 'cyan', 'fillColor': 'cyan', 'fillOpacity': 0.4, 'weight': 2}, layer_name='OSM Wetland Boundary')
MapOSM.addLayer(geometry, {'color': 'red'}, 'Study Area', opacity=0.5)
MapOSM

In [ ]:
# Rasterize OSM polygons to match Sentinel raster grid

# Get reference raster properties from an existing S1 file
ref_path = sorted(S1_OUTPUT_DIR.glob('S1_VV_*.tif'))[0]
with rasterio.open(ref_path) as src:
    ref_transform = src.transform
    ref_shape = (src.height, src.width)
    ref_crs = src.crs
    ref_profile = src.profile.copy()

print(f"Reference raster: {ref_shape[0]} x {ref_shape[1]}, CRS: {ref_crs}")

# Reproject OSM polygons from EPSG:4326 (WGS84) to raster CRS (EPSG:32647)
transformer = Transformer.from_crs("EPSG:4326", str(ref_crs), always_xy=True)

def reproject_polygon(poly):
    ext = [transformer.transform(x, y) for x, y in poly.exterior.coords]
    ints = [[transformer.transform(x, y) for x, y in ring.coords] for ring in poly.interiors]
    return Polygon(ext, ints)

if wetland_geom.geom_type == 'MultiPolygon':
    projected = MultiPolygon([reproject_polygon(p) for p in wetland_geom.geoms])
elif wetland_geom.geom_type == 'Polygon':
    projected = reproject_polygon(wetland_geom)

# Rasterize: 1 = wetland, 0 = non-wetland
mask_arr = rio_rasterize(
    [(projected, 1)],
    out_shape=ref_shape,
    transform=ref_transform,
    fill=0,
    dtype='uint8'
)

print(f"Wetland pixels: {mask_arr.sum():,} / {mask_arr.size:,} ({mask_arr.sum()/mask_arr.size*100:.1f}%)")

# Save as GeoTIFF (overwrites old mask)
wetland_mask_path = Path('data/raw') / 'wetland_mask.tif'
ref_profile.update(count=1, dtype='uint8', nodata=None)

with rasterio.open(wetland_mask_path, 'w', **ref_profile) as dst:
    dst.write(mask_arr, 1)

print(f"Saved: {wetland_mask_path}")

## Next Steps

1. Verify all monthly GeoTIFFs are present (check output above)
2. Re-run any failed months if needed
3. Proceed to `02_data_preprocessing.ipynb` to build the pixel-level data cubes